---
---

# Part 2: 데이터 표준화 (Standardization)

## human_data.csv — Height, Weight, Foot size 표준화 시각화

### 표준화란 무엇인가?

표준화(Standardization)는 각 변수를 **평균 0, 표준편차 1**인 분포로 변환하는 기법입니다.

$$z = \frac{x - \mu}{\sigma}$$

- $x$: 원본 값
- $\mu$: 해당 변수의 평균
- $\sigma$: 해당 변수의 표준편차
- $z$: 변환된 값 (Z-score)

### 왜 표준화가 필요한가?

머신러닝 알고리즘의 대부분은 **거리 기반(distance-based)** 계산을 수행합니다:
- KNN: 유클리드 거리로 이웃 결정
- SVM: 초평면까지의 거리 최적화
- 경사 하강법: 기울기(gradient) 기반 학습

변수의 스케일이 다르면:
- **Height**: ~5.0~6.5 (작은 범위)
- **Weight**: ~100~200 (큰 범위)
- **Foot size**: ~6~13 (중간 범위)

→ Weight의 숫자가 크기 때문에 거리 계산에서 **Weight만 지배적 영향**을 미침
→ Height와 Foot size의 정보가 사실상 **무시**됨
→ 이것은 알고리즘의 오류가 아니라, **입력 데이터의 스케일 불균형** 문제입니다.

### 모든 변수의 값이 동일해지는 것 아닌가?

**아닙니다.** 이것은 가장 흔한 오해입니다.

표준화는 다음을 **변경합니다**:
- 중심(center): 모든 변수의 평균 → 0
- 스케일(scale): 모든 변수의 표준편차 → 1

표준화는 다음을 **보존합니다**:
- **분포 형태(shape)**: 원래 정규분포면 여전히 정규분포, 비대칭이면 여전히 비대칭
- **상대적 순서(rank)**: A가 B보다 컸으면 변환 후에도 A > B
- **이상치(outlier)**: 원래 극단값이면 변환 후에도 극단값
- **변수 간 상관관계**: Height와 Weight의 상관계수 유지

비유하면: **온도를 섭씨에서 화씨로 바꿔도 "오늘이 어제보다 덥다"는 사실은 변하지 않는 것**과 같습니다.

아래 히스토그램에서 이를 직접 확인하겠습니다.

In [ ]:
# GitHub에서 human_data.csv 로드
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

url = "https://raw.githubusercontent.com/ancestor9/data/main/human_data.csv"
hdf = pd.read_csv(url)

print(f"데이터 형태: {hdf.shape}")
print(f"\n컬럼: {list(hdf.columns)}")
print()
hdf.head(10)

In [ ]:
# 원본 데이터 기술통계 — 스케일 차이 확인
print("=== 원본 데이터 기술통계 ===")
print(hdf[["Height", "Weight", "Foot size"]].describe())
print("\n주목: Height(~5.5), Weight(~150), Foot size(~9) — 스케일이 극단적으로 다름")

### 표준화 전 히스토그램

세 변수를 **같은 X축**에 그리면 스케일 차이가 어떤 문제를 일으키는지 확인할 수 있습니다.
- Weight(빨간색)가 X축의 100~200 구간을 차지
- Height(파란색)와 Foot size(초록색)는 0~15 근처에 **뭉개져서** 거의 보이지 않음

이 상태로 KNN, SVM 등 거리 기반 모델에 입력하면 Weight만으로 결과가 결정됩니다.

In [ ]:
# 표준화 전 히스토그램
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(hdf["Height"], bins=20, alpha=0.7, color="blue", label="Height", edgecolor="black")
ax.hist(hdf["Weight"], bins=20, alpha=0.7, color="red", label="Weight", edgecolor="black")
ax.hist(hdf["Foot size"], bins=20, alpha=0.7, color="green", label="Foot size", edgecolor="black")

ax.set_title("표준화 전: 원본 데이터 히스토그램", fontsize=16, fontweight="bold")
ax.set_xlabel("값", fontsize=13)
ax.set_ylabel("빈도", fontsize=13)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ Weight(빨간색)가 X축을 지배하여 Height(파란색)와 Foot size(초록색)가 거의 보이지 않음")
print("→ 이 상태로 거리 기반 모델에 입력하면 Weight만 결과에 영향을 미침")

In [ ]:
# StandardScaler로 표준화 적용
scaler = StandardScaler()
features = hdf[["Height", "Weight", "Foot size"]]

scaled_values = scaler.fit_transform(features)
scaled_df = pd.DataFrame(scaled_values, columns=["Height", "Weight", "Foot size"])

print("=== 표준화 후 기술통계 ===")
print(scaled_df.describe())
print("\n주목:")
print("  - 모든 변수의 평균(mean) ≈ 0")
print("  - 모든 변수의 표준편차(std) ≈ 1")
print("  - 그러나 min, max, 사분위수는 변수마다 다름 → 분포 형태가 보존됨!")

### 표준화 후 히스토그램

표준화 후 세 변수를 같은 X축에 그리면:
- 세 변수 모두 **0 근처에 중심**이 위치
- 대략 **-3 ~ +3 범위**에 데이터가 분포
- **각 변수의 분포 형태(shape)는 원본과 동일** — 이것이 핵심!

이제 거리 기반 모델에서 세 변수가 **동등한 영향력**으로 기여합니다.

In [ ]:
# 표준화 후 히스토그램
fig, ax = plt.subplots(figsize=(12, 6))

ax.hist(scaled_df["Height"], bins=20, alpha=0.7, color="blue", label="Height (표준화)", edgecolor="black")
ax.hist(scaled_df["Weight"], bins=20, alpha=0.7, color="red", label="Weight (표준화)", edgecolor="black")
ax.hist(scaled_df["Foot size"], bins=20, alpha=0.7, color="green", label="Foot size (표준화)", edgecolor="black")

ax.set_title("표준화 후: Z-score 변환 히스토그램", fontsize=16, fontweight="bold")
ax.set_xlabel("Z-score (표준화된 값)", fontsize=13)
ax.set_ylabel("빈도", fontsize=13)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ 세 변수 모두 동일한 축 범위(-3~+3)에서 비교 가능")
print("→ 각 변수의 분포 형태(shape)는 원본과 동일하게 보존됨")
print("→ 표준화는 \"값을 같게 만드는 것\"이 아니라 \"스케일을 통일하는 것\"입니다")

---
### 과학적 정리: 표준화는 왜 값을 동일하게 만들지 않는가?

#### 수학적 증명

표준화 공식 $z = \frac{x - \mu}{\sigma}$는 **선형 변환(affine transformation)** 입니다:
- 평행이동(translation): $x - \mu$ → 분포를 원점으로 이동
- 스케일링(scaling): $\div \sigma$ → 퍼짐 정도를 1로 조정

선형 변환의 핵심 성질:
1. **순서 보존**: $x_1 > x_2 \Rightarrow z_1 > z_2$ (항상 성립, $\sigma > 0$이므로)
2. **분포 형태 보존**: 원래 데이터의 왜도(skewness), 첨도(kurtosis) 불변
3. **상관관계 보존**: $\text{corr}(X, Y) = \text{corr}(Z_X, Z_Y)$

#### 직관적 이해

| 변환 전 | 변환 후 | 의미 |
|---------|---------|------|
| Height = 6.0 (평균 5.5보다 큼) | Z = +1.2 (평균보다 1.2σ 위) | **상대적 위치 동일** |
| Weight = 180 (평균 150보다 큼) | Z = +1.5 (평균보다 1.5σ 위) | **상대적 위치 동일** |

Z-score는 "이 값이 평균에서 표준편차 몇 개 만큼 떨어져 있는가"를 나타냅니다.
**단위가 다른 변수들을 "표준편차" 라는 공통 단위로 환산**한 것입니다.

#### 비유

수학 시험 80점, 영어 시험 90점 — 어느 과목을 더 잘한 것인가?
- 수학 평균 60, 표준편차 10 → Z = (80-60)/10 = **+2.0**
- 영어 평균 85, 표준편차 5 → Z = (90-85)/5 = **+1.0**
- → 수학을 상대적으로 **더 잘함** (평균 대비 2σ vs 1σ)

이처럼 표준화는 **절대값을 동일하게 만드는 것이 아니라**, 서로 다른 척도를 **공정하게 비교 가능한 공통 기준**으로 변환하는 것입니다.